In [1]:
import pandas as pd
import numpy as np
import os
import pyreadstat

# Base

In [2]:
BD = pd.read_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\data\BD_Ascenso24_MINEDU_MINDEF_v200526.xlsx",
              sheet_name="BD_MINEDU_MINDEF"
              )

In [3]:
BD["BASE"].value_counts(dropna=False)

BASE
MINEDU    89754
MINDEF      178
Name: count, dtype: int64

# Tablas

### T1. Participación docente por etapas 

In [172]:
resumen = pd.DataFrame({
    "Inscritos": [
        BD[BD['BASE'] == 'MINEDU'].shape[0],
        BD[BD['BASE'] == 'MINDEF'].shape[0]
    ],
    "Evaluados Etapa Nac.": [
        BD[
            (BD['BASE'] == 'MINEDU') &
            (BD['ESTADO_PUN'].isin(['SUPERÓ PUNTAJE MÍNIMO', 'NO SUPERÓ PUNTAJE MÍNIMO']))
        ].shape[0],
        BD[
            (BD['BASE'] == 'MINDEF') &
            (BD['ESTADO_PUN'].isin(['SUPERÓ PUNTAJE MÍNIMO', 'NO SUPERÓ PUNTAJE MÍNIMO']))
        ].shape[0]
    ],
    "Evaluados Etapa Des.": [
        BD[
            (BD['BASE'] == 'MINEDU') &
            (BD['CONDICION_FINAL'].isin(['HABILITADO PARA ASIGNACIÓN DE VACANTE', 'NO CUMPLE REQUISITOS']))
        ].shape[0],
        BD[
            (BD['BASE'] == 'MINDEF') &
            (BD['CONDICION_FINAL'].isin(['HABILITADO PARA ASIGNACIÓN DE VACANTE', 'NO CUMPLE REQUISITOS']))
        ].shape[0]
    ],
    "Ganadores": [
        BD[(BD['BASE'] == 'MINEDU') & (BD['GANADOR_ASCENSO'] == 1)].shape[0],
        BD[(BD['BASE'] == 'MINDEF') & (BD['GANADOR_ASCENSO'] == 1)].shape[0]
    ]
}, 
index=['Minedu', 'Mindef'])

# Fila Total
resumen.loc['Total'] = resumen.sum()

resumen.reset_index(inplace=True)
T1 = resumen.copy()

T1.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T1.xlsx", 
            index=False)

### T2. Evaluados y aprobados de la Etapa Nacional, y ganadores por escala magisterial 

In [174]:
orden_escalas = [
    "Segunda Escala",
    "Tercera Escala",
    "Cuarta Escala",
    "Quinta Escala",
    "Sexta Escala",
    "Séptima Escala",
    "Octava Escala"
]

def conteo(mask, nombre):
    return (
        BD.loc[mask, "ESCALA_POSTULA_DESCRIPCION"]
        .value_counts()
        .reindex(orden_escalas, fill_value=0)
        .rename(nombre)
    )

tabla = pd.concat([
    conteo(
        BD["ESTADO_PUN"].isin([
            "SUPERÓ PUNTAJE MÍNIMO",
            "NO SUPERÓ PUNTAJE MÍNIMO"
        ]),
        "Evaluados Etapa Nac."
    ),
    conteo(
        BD["ESTADO_PUN"].eq("SUPERÓ PUNTAJE MÍNIMO"),
        "Aprobados Etapa Nac."
    ),
    conteo(
        BD["GANADOR_ASCENSO"].eq(1),
        "Ganadores"
    )
], axis=1)

tabla.loc["Total"] = tabla.sum()
tabla.reset_index(inplace=True)
T2 = tabla.copy()
T2["Ganadores/Evaluados"] = round(T2["Ganadores"] / T2["Evaluados Etapa Nac."] * 100, 0)
T2
T2.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T2.xlsx", 
            index=False)

### T3. Ganadores por sexo

In [175]:
resultado = (
    BD.query("GANADOR_ASCENSO == 1")
    .groupby(["BASE", "SEXO_POSTULANTE"])
    .size()
    .unstack(fill_value=0)
    .assign(
        Total=lambda x: x.sum(axis=1),
        Mujeres_pct=lambda x: round(x["FEMENINO"] / x["Total"]*100,0),
        Varones_pct=lambda x: round(x["MASCULINO"] / x["Total"]*100,0)
    )
    .pipe(
        lambda x: pd.concat([
            x,
            pd.DataFrame({
                "FEMENINO": [x["FEMENINO"].sum()],
                "MASCULINO": [x["MASCULINO"].sum()],
                "Total": [x["Total"].sum()],
                "Mujeres_pct": [round(x["FEMENINO"].sum() / x["Total"].sum()*100,0)],
                "Varones_pct": [round(x["MASCULINO"].sum() / x["Total"].sum()*100,0)]
            }, index=["Total"])
        ])
    )
).reset_index()

T3 = resultado.copy()
T3

T3.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T3.xlsx", 
            index=False)

### T4. Ganadores segun rango de edad

In [176]:
orden_escalas = [
    "25-29",
    "30-34",
    "35-39",
    "40-44",
    "45-49",
    "50-54",
    "55-59",
    "60 a más"
]

def conteo(mask, nombre):
    return (
        BD.loc[mask, "RANGO_EDAD_IDR"]
        .value_counts()
        .reindex(orden_escalas, fill_value=0)
        .rename(nombre)
    )

tabla = (
    pd.concat([
        conteo(
            BD["ESTADO_PUN"].isin([
                "SUPERÓ PUNTAJE MÍNIMO",
                "NO SUPERÓ PUNTAJE MÍNIMO"
            ]),
            "Evaluados Etapa Nac."
        ),
        conteo(
            BD["GANADOR_ASCENSO"].eq(1),
            "Ganadores"
        )
    ], axis=1)
    .assign(
        **{
            "%_Ganadores": lambda x: round(
                x["Ganadores"] / x["Ganadores"].sum() * 100,
                0
            )
        }
    )
)

tabla.loc["Total"] = [
    tabla["Evaluados Etapa Nac."].sum(),
    tabla["Ganadores"].sum(),
    tabla["%_Ganadores"].sum()  # Esto sumará los porcentajes, lo cual no es correcto, pero se deja para mantener la estructura
]

tabla.reset_index(inplace=True)
tabla.rename(columns={"index": "RANGO_EDAD_IDR"}, inplace=True)

T4 = tabla.copy()
T4["Ganadores_Evaluados"] = round(T4["Ganadores"] / T4["Evaluados Etapa Nac."] * 100, 0)
T4

T4.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T4.xlsx", 
            index=False)

### T5. Ganadores según discapacidad

In [107]:
BD["RECIBIO_BONIF_DISPAC_IDR"] = np.where(
    BD["BONIFICACION_DISCAPACIDAD_A"] != 0,
    "Sí",
    "No"
)

In [178]:
T5 = pd.concat([
    BD.loc[
        BD["ESTADO_PUN"].isin([
            "SUPERÓ PUNTAJE MÍNIMO",
            "NO SUPERÓ PUNTAJE MÍNIMO"
        ]),
        "TIENE_DISCAPACIDAD"
    ]
    .value_counts()
    .rename("Evaluados"),

    BD.loc[
        BD["GANADOR_ASCENSO"].eq(1),
        "RECIBIO_BONIF_DISPAC_IDR"
    ]
    .value_counts()
    .rename("Ganadores")
], axis=1)\
        .pipe(
            lambda x: x.assign(**{"Ganadores_%": round(x["Ganadores"] / x["Ganadores"].sum() * 100, 0)})
        )

T5.index = ["Sin discapacidad", "Con discapacidad"]

T5.loc["Total"] = T5.sum()
T5["Ganadores_sobre_Evaluados"] = round(T5["Ganadores"] / T5["Evaluados"] * 100, 0)

T5.reset_index(inplace=True)
T5.rename(columns={"index": "Condición de discapacidad"}, inplace=True)

T5

T5.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T5.xlsx", 
            index=False)

### T6_A y T6_B

In [180]:
tabla = (
    BD.groupby(["BASE", "REGION_LEGAJO_ACTUALIZADO"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla.loc["Total"] = [
    "",
    "TOTAL",
    tabla["Evaluados_PUN"].sum(),
    tabla["Aprobados_PUN"].sum(),
    tabla["Reprobados_PUN"].sum(),
    round(
        tabla["Aprobados_PUN"].sum()
        / tabla["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla

tabla.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T6_A.xlsx", 
            index=False)

In [181]:
tabla = (
    BD.groupby(["BASE", "REGION_LEGAJO_ACTUALIZADO"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla.loc["Total"] = [
    "",
    "TOTAL",
    tabla["Evaluados_PUN"].sum(),
    tabla["Aprobados_PUN"].sum(),
    tabla["Reprobados_PUN"].sum(),
    round(
        tabla["Aprobados_PUN"].sum()
        / tabla["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla_2 = (
    BD.groupby(["BASE", "DESC_UNIDAD_EJECUTORA"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla_2.loc["Total"] = [
    "",
    "TOTAL",
    tabla_2["Evaluados_PUN"].sum(),
    tabla_2["Aprobados_PUN"].sum(),
    tabla_2["Reprobados_PUN"].sum(),
    round(
        tabla_2["Aprobados_PUN"].sum()
        / tabla_2["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla_2

tabla_2.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T6_B.xlsx", 
            index=False)

### T7

In [182]:
tabla = (
    BD.groupby(["GRUPO_INSCRIPCION_FINAL"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla.loc["Total"] = [
    "TOTAL",
    tabla["Evaluados_PUN"].sum(),
    tabla["Aprobados_PUN"].sum(),
    tabla["Reprobados_PUN"].sum(),
    round(
        tabla["Aprobados_PUN"].sum()
        / tabla["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla

tabla.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T7.xlsx", 
            index=False)

### T8

In [183]:
BD["TRAYECTORIA_1_1_MASTER"] = np.where(BD["TRAYECTORIA_1_1_GRADO"] == "Grado de Maestro/Maestría/Magíster/Máster", "Sí", "No")
BD["TRAYECTORIA_1_1_DOCTOR"] = np.where(BD["TRAYECTORIA_1_1_GRADO"] == "Grado de Doctor", "Sí", "No")

In [184]:
COLUMNAS_ED = [
    'TRAYECTORIA_1_1_MASTER',
    'TRAYECTORIA_1_1_DOCTOR',
    'TRAYECTORIA_1_2',
    'TRAYECTORIA_1_3',
    'TRAYECTORIA_1_4',
    'TRAYECTORIA_1_5',
    'TRAYECTORIA_2_1',
    'TRAYECTORIA_2_2',
    'TRAYECTORIA_2_3',
    'TRAYECTORIA_2_4',
    'TRAYECTORIA_2_5A',
    'TRAYECTORIA_2_5B',
    'TRAYECTORIA_2_5C',
    'TRAYECTORIA_2_5D',
    'TRAYECTORIA_3_1A',
    'TRAYECTORIA_3_1B',
    'TRAYECTORIA_3_1C',
    'TRAYECTORIA_3_2',
    'TRAYECTORIA_3_3',
    'TRAYECTORIA_3_4',
    'TRAYECTORIA_3_5',
    'TRAYECTORIA_3_6'
]

T8 = (
    BD.loc[
        BD["CONDICION_FINAL"].isin([
            "HABILITADO PARA ASIGNACIÓN DE VACANTE",
            "NO CUMPLE REQUISITOS"
        ]),
        COLUMNAS_ED
    ]
    .eq("Sí")
    .sum()
    .reset_index(name="Cumplieron")
    .rename(columns={"index": "Criterio"})
    .assign(
        Evaluados=lambda x: BD["CONDICION_FINAL"].isin([
            "HABILITADO PARA ASIGNACIÓN DE VACANTE",
            "NO CUMPLE REQUISITOS"
        ]).sum(),
        Criterio=lambda x: (
            x["Criterio"]
            .str.replace("TRAYECTORIA_", "Criterio ", regex=False)
            .str.replace("_GRADO", "a", regex=False)
            .str.replace("_", ".", regex=False)
            .str.lower()
            .str.replace(".5a", ".5a")
            .str.replace(".5b", ".5b")
            .str.replace(".5c", ".5c")
            .str.replace(".5d", ".5d")
            .str.title()
        )
    )
    [["Criterio", "Evaluados", "Cumplieron"]]
)

T8["Cumplimiento_pct"] = round(T8["Cumplieron"] / T8["Evaluados"] * 100, 0)

T8.to_excel(r"C:\Users\User\OneDrive - Universidad Torcuato Di Tella\2. PC TUF reforzada\MINEDU 2026\Repositorios\MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024\output\T8.xlsx", 
            index=False)